# 10 - CASMI v2 Colab setup, environment and pipeline state

**Objective.** Verify that this runtime can run the v2 protocol notebooks, report where the pipeline stands and which notebook to run next. **Nothing is trained or rebuilt here.**

| layer | location | rule |
|---|---|---|
| source | GitHub `mybenkhadda/enveda-kaggle` -> `/content/Enveda` | cloned if missing, otherwise fetched + fast-forwarded by the bootstrap cell; never copied from Drive |
| persistence | `/content/drive/MyDrive/EnvedaCASMI` | inputs, frozen bundle, every artifact |
| scratch | `/content/enveda_work` (local SSD) | staged copies, temporaries; disposable |

Every v2 notebook starts with the same two cells:

1. the **generated bootstrap cell** (`casmi.workspace.notebook_cells`, stamped by `scripts/stamp_notebooks.py`): mounts Drive, sets `ENVEDA_DRIVE_ROOT` / `ENVEDA_REPO_ROOT`, clones or fast-forwards the repository, drops already-imported `casmi` modules, puts `src/` on `sys.path`;
2. `bootstrap(...)` (`casmi.workspace.bootstrap`): checks the notebook / bootstrap-cell / config / path / artifact-registry versions, loads the config, builds `P` (V2Paths, the physical roots) and `ARTIFACTS` (ArtifactRegistry, every semantic location) and runs **preflight** -- any incompatibility or missing upstream artifact stops the notebook before work starts, with the fix in the message.

**Run order**

* first setup: **10 -> 11 (preliminary) -> 12**
* after a real external candidate source has been ingested by 12: **11 with `REBUILD = True` -> 13 (Gate A)**
* only if Gate A is **protocol-valid**: **14** (analog baseline)

A TRAIN-only candidate universe is **not** a valid C2 (external-universe) evaluation. Private repository: add a Colab secret `GITHUB_TOKEN`.

In [ ]:
# >>> CASMI BOOTSTRAP casmi-v2-bootstrap-2 (generated by scripts/stamp_notebooks.py; do not edit by hand)
# mount Drive -> env roots -> clone if missing, else fetch + checkout + pull --ff-only -> drop stale casmi modules -> src on sys.path
import importlib, os, subprocess, sys
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
REPO_ROOT = Path(os.environ.setdefault('ENVEDA_REPO_ROOT', '/content/Enveda'))
os.environ.setdefault('ENVEDA_SCRATCH_DIR', '/content/enveda_work')
BRANCH = os.environ.setdefault('ENVEDA_BRANCH', 'main')
SYNC_REPO = os.environ.get('ENVEDA_SYNC_REPO', '1' if IN_COLAB else '0') == '1'  # outside Colab the working tree is used as is
GITHUB_REPO = 'mybenkhadda/enveda-kaggle'


def _git(*args):
    r = subprocess.run(['git', '-C', str(REPO_ROOT), *args], capture_output=True, text=True)
    if r.returncode != 0:
        cmd = ' '.join(args)
        raise RuntimeError(f'git {cmd} failed in {REPO_ROOT}: {r.stderr.strip()} | Fix: discard local edits inside the clone '
                           f'(git -C {REPO_ROOT} stash) or delete {REPO_ROOT}, then re-run this cell; source lives on GitHub only.')
    return r.stdout.strip()


if SYNC_REPO:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')  # optional Colab secret (private repository)
        except Exception:
            token = None
    url = f'https://{token}@github.com/{GITHUB_REPO}.git' if token else f'https://github.com/{GITHUB_REPO}.git'
    if not (REPO_ROOT / '.git').exists():
        r = subprocess.run(['git', 'clone', '--branch', BRANCH, url, str(REPO_ROOT)], capture_output=True, text=True)
        if r.returncode != 0:
            err = r.stderr.replace(token, '***') if token else r.stderr
            raise RuntimeError(f'git clone of {GITHUB_REPO} failed: {err.strip()} | Fix: check the GITHUB_TOKEN Colab secret / network.')
    else:
        _git('fetch', 'origin', BRANCH)
        _git('checkout', BRANCH)
        _git('pull', '--ff-only', 'origin', BRANCH)
if not (REPO_ROOT / 'src' / 'casmi' / 'workspace' / 'versions.py').exists():
    raise RuntimeError(f'{REPO_ROOT} has no src/casmi/workspace/versions.py: the source is OLDER than this notebook. '
                       f'Fix: push the bootstrap/artifact-registry refactor to GitHub {BRANCH}, restart the runtime, re-run this cell.')
for _m in [m for m in list(sys.modules) if m == 'casmi' or m.startswith('casmi.')]:
    del sys.modules[_m]  # after a pull: never mix old and new source in one kernel
importlib.invalidate_caches()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.environ['ENVEDA_BOOTSTRAP_CELL'] = 'casmi-v2-bootstrap-2'
_head = _git('log', '-1', '--format=%h %s') if (REPO_ROOT / '.git').exists() else 'no git (uploaded copy)'
print(f'source {REPO_ROOT} @ {_head} | drive {DRIVE_ROOT} | colab {IN_COLAB} | synced {SYNC_REPO}')

In [ ]:
from casmi.workspace.bootstrap import CHEM_PACKAGES, RANK_PACKAGES, bootstrap

NOTEBOOK = '10_colab_v2_setup'
CTX = bootstrap(NOTEBOOK, notebook_api='casmi-v2-notebooks-4',
                uses_paths=('drive_root', 'repo_root', 'scratch_dir'),
                uses_artifacts=('copy_manifest', 'external_coconut_dir', 'environment_reports_dir'),
                requires=('train_spectrum_metadata', 'connectivity_folds', 'structure_table', 'dev_queries', 'molecule_mass_variants',
                          'bundle_config'),
                packages={**CHEM_PACKAGES, **RANK_PACKAGES}, check_bundle=True)
CFG, P, ARTIFACTS = CTX.cfg, CTX.paths, CTX.artifacts

## 1. Drive inputs and frozen bundle identity

The five canonical inputs live in `data/processed/` on Drive (copied by `scripts/copy_colab_assets_to_drive.ps1`, see `docs/COLAB_DRIVE_MIGRATION.md`). The frozen v2-A7 bundle is identified by its **config labels** (`bundle_version`, `CONFIG_HASH`, `model_id`): `CONFIG_HASH` is an identity label and no file is hashed (no-hash policy).

In [ ]:
from casmi.workspace.artifact_registry import FROZEN_BUNDLE_IDENTITY, bundle_identity, check_bundle_identity
from casmi.workspace.colab_paths import missing_required

MISSING = missing_required(P)
for m in MISSING:
    print('MISSING', m)
if MISSING:
    raise FileNotFoundError(f'{len(MISSING)} required Drive inputs missing -- run the Windows copy (docs/COLAB_DRIVE_MIGRATION.md) and wait for Drive sync')
IDENTITY = bundle_identity(ARTIFACTS)
BUNDLE_PROBLEMS = check_bundle_identity(ARTIFACTS)
assert not BUNDLE_PROBLEMS, BUNDLE_PROBLEMS
assert IDENTITY == FROZEN_BUNDLE_IDENTITY
print('copy manifest:', ARTIFACTS.copy_manifest if ARTIFACTS.copy_manifest.exists() else 'not found (migration not run yet?)')
print('Drive inputs OK | frozen bundle identity labels:', IDENTITY)

## 2. External candidate sources

The Class-2 protocol needs at least one **non-TRAIN** candidate source. COCONUT is expected at `data/external/coconut/coconut.csv` on Drive (column mapping: `configs/v6/coconut_source.json`); PubChem subsets go to `data/external/pubchem/`. Nothing is ever downloaded by code.

In [ ]:
from casmi.workspace.preflight import external_sources_table

SOURCES = external_sources_table(CFG, P)
display(SOURCES)
if not SOURCES['exists'].any():
    print('NO external candidate source on Drive. Expected COCONUT at', ARTIFACTS.external_coconut_dir / 'coconut.csv')
    print('Until it exists, notebook 12 builds a TRAIN-only universe and every C2 result is PRELIMINARY / protocol-invalid.')

## 3. Pipeline state

Every artifact location comes from the registry (`casmi.workspace.artifact_registry`). `pipeline_status` reads small JSON manifests only and flags stale stages (regimes built for another universe), a TRAIN-only universe (C2 protocol invalid) and a protocol-invalid Gate A (notebook 14 blocked).

In [ ]:
import pandas as pd
from casmi.workspace.artifact_registry import pipeline_status

with pd.option_context('display.max_colwidth', 140, 'display.width', 240):
    display(ARTIFACTS.table()[['name', 'exists', 'producer', 'path']])
    STATUS, NEXT = pipeline_status(CFG, P)
    display(STATUS)
print('NEXT NOTEBOOK:', NEXT)

## 4. Accelerator

Notebooks 11-13 are CPU-only; 14 uses CUDA for the sparse prefilter if available. AMP: bf16 only where supported, otherwise fp16 with a GradScaler.

In [ ]:
import json
from casmi.workspace.environment import choose_amp_dtype, gpu_report, recommend_batch_size, system_report

GPU, SYS = gpu_report(), system_report()
AMP_DTYPE = choose_amp_dtype(CFG['runtime']['amp'], GPU)
print(json.dumps(GPU, indent=2))
print('AMP dtype:', AMP_DTYPE, '| example batch sizes:', {t: recommend_batch_size(t, GPU) for t in ('fingerprint', 'contrastive')})
if not GPU['cuda_available']:
    print('NOTE: no CUDA GPU. Notebooks 11-14 run on CPU.')

## 5. Optional self-tests

The fast unit tests that guard this pipeline (path API, artifact registry, preflight, C2 protocol, cache identity, tracker, scorer, notebook auditor) plus the static notebook audit. A failure means the source on this runtime is inconsistent: fix the source, never the notebook. Set `RUN_SELF_TESTS = False` to skip.

In [ ]:
RUN_SELF_TESTS = True
FAST_TESTS = ['tests/test_v2_workspace.py', 'tests/test_artifact_registry.py', 'tests/test_preflight.py', 'tests/test_c2_protocol.py',
              'tests/test_cache_identity.py', 'tests/test_notebook_audit.py', 'tests/test_official_metric.py', 'tests/test_v2_regimes.py',
              'tests/test_v2_recall.py', 'tests/test_v2_metrics.py']
if RUN_SELF_TESTS:
    r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', *FAST_TESTS], cwd=P.repo_root,
                       capture_output=True, text=True, env={**os.environ, 'PYTHONPATH': str(P.repo_root / 'src')})
    print(r.stdout[-3000:], r.stderr[-2000:])
    assert r.returncode == 0, 'self-tests failed -- the source on this runtime is inconsistent; fix the source, never the notebook'
    r = subprocess.run([sys.executable, 'scripts/audit_notebooks.py', '--quiet'], cwd=P.repo_root, capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-1000:])
    assert r.returncode == 0, 'notebook audit failed: python scripts/audit_notebooks.py lists the problems'

## 6. Environment record and run order

In [ ]:
from datetime import datetime, timezone
from casmi.workspace.bootstrap import write_json

STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
env = {**CTX.metadata(), 'created_at': datetime.now(timezone.utc).isoformat(), 'gpu': GPU, 'system': SYS, 'amp_dtype': AMP_DTYPE,
       'bundle_identity': IDENTITY, 'paths': P.as_dict(), 'external_sources': SOURCES.to_dict(orient='records'),
       'pipeline_status': STATUS.to_dict(orient='records'), 'next_notebook': NEXT}
OUT = write_json(ARTIFACTS.environment_reports_dir / f'environment_{STAMP}.json', env)
RUN_ORDER = ['first setup:                           10 -> 11 (preliminary) -> 12',
             'after a real external universe exists: 11 (REBUILD = True) -> 13 (Gate A)',
             'only if Gate A is protocol-valid:      14 (analog baseline)']
print('READY | source', CTX.git_commit, '| drive', P.drive_root, '| scratch', P.scratch_dir)
print('record', OUT)
for line in RUN_ORDER:
    print(line)
print('NEXT NOTEBOOK:', NEXT)